In [0]:
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, IntegerType
)

bronze_schema = StructType([
    StructField("indicator", StructType([
        StructField("id", StringType(), True),
        StructField("value", StringType(), True)
    ]), True),
    StructField("country", StructType([
        StructField("id", StringType(), True),
        StructField("value", StringType(), True)
    ]), True),
    StructField("countryiso3code", StringType(), True),
    StructField("date", StringType(), True),
    StructField("value", DoubleType(), True),
    StructField("unit", StringType(), True),
    StructField("obs_status", StringType(), True),
    StructField("decimal", StringType(), True)
])

In [0]:
bronze_df = (
    spark.read
    .schema(bronze_schema)
    .option("multiline", "true")
    .json("/Workspace/Users/l242585@lhr.nu.edu.pk/World_Bank_Analytics/full_load_part1.json")
)

# Agar teeno parts combine karni hain
bronze_df2 = spark.read.schema(bronze_schema).option("multiline", "true").json("/Workspace/Users/l242585@lhr.nu.edu.pk/World_Bank_Analytics/full_load_part1.json/")
bronze_df3 = spark.read.schema(bronze_schema).option("multiline", "true").json("/Workspace/Users/l242585@lhr.nu.edu.pk/World_Bank_Analytics/full_load_part1.json")

bronze_df = bronze_df.unionByName(bronze_df2).unionByName(bronze_df3)

print(f"Bronze row count: {bronze_df.count()}")
bronze_df.printSchema()

Bronze row count: 907227
root
 |-- indicator: struct (nullable = true)
 |    |-- id: string (nullable = true)
 |    |-- value: string (nullable = true)
 |-- country: struct (nullable = true)
 |    |-- id: string (nullable = true)
 |    |-- value: string (nullable = true)
 |-- countryiso3code: string (nullable = true)
 |-- date: string (nullable = true)
 |-- value: double (nullable = true)
 |-- unit: string (nullable = true)
 |-- obs_status: string (nullable = true)
 |-- decimal: string (nullable = true)



In [0]:
from pyspark.sql.functions import current_timestamp

bronze_df = bronze_df.withColumn("load_timestamp", current_timestamp())
bronze_df.display()

indicator,country,countryiso3code,date,value,unit,obs_status,decimal,load_timestamp
"List(AG.CON.FERT.PT.ZS, Fertilizer consumption (% of fertilizer production))","List(ZH, Africa Eastern and Southern)",AFE,2024,null,,,1,2026-10-05T19:38:25.027Z
"List(AG.CON.FERT.PT.ZS, Fertilizer consumption (% of fertilizer production))","List(ZH, Africa Eastern and Southern)",AFE,2023,1716.65469673153,,,1,2026-10-05T19:38:25.027Z
"List(AG.CON.FERT.PT.ZS, Fertilizer consumption (% of fertilizer production))","List(ZH, Africa Eastern and Southern)",AFE,2022,793.122757182063,,,1,2026-10-05T19:38:25.027Z
"List(AG.CON.FERT.PT.ZS, Fertilizer consumption (% of fertilizer production))","List(ZH, Africa Eastern and Southern)",AFE,2021,869.682412413707,,,1,2026-10-05T19:38:25.027Z
"List(AG.CON.FERT.PT.ZS, Fertilizer consumption (% of fertilizer production))","List(ZH, Africa Eastern and Southern)",AFE,2020,910.519922715408,,,1,2026-10-05T19:38:25.027Z
"List(AG.CON.FERT.PT.ZS, Fertilizer consumption (% of fertilizer production))","List(ZH, Africa Eastern and Southern)",AFE,2019,831.511420068027,,,1,2026-10-05T19:38:25.027Z
"List(AG.CON.FERT.PT.ZS, Fertilizer consumption (% of fertilizer production))","List(ZH, Africa Eastern and Southern)",AFE,2018,678.189733859044,,,1,2026-10-05T19:38:25.027Z
"List(AG.CON.FERT.PT.ZS, Fertilizer consumption (% of fertilizer production))","List(ZH, Africa Eastern and Southern)",AFE,2017,752.709511568123,,,1,2026-10-05T19:38:25.027Z
"List(AG.CON.FERT.PT.ZS, Fertilizer consumption (% of fertilizer production))","List(ZH, Africa Eastern and Southern)",AFE,2016,708.291488756049,,,1,2026-10-05T19:38:25.027Z
"List(AG.CON.FERT.PT.ZS, Fertilizer consumption (% of fertilizer production))","List(ZH, Africa Eastern and Southern)",AFE,2015,658.889082780502,,,1,2026-10-05T19:38:25.027Z


In [0]:
bronze_df.write.format("delta").mode("overwrite").saveAsTable("bronze_indicator_values")

In [0]:
from pyspark.sql.functions import col, when

silver_df = (
    bronze_df
    .select(
        col("country.id").alias("country_code"),
        col("country.value").alias("country_name"),
        col("indicator.id").alias("indicator_code"),
        col("indicator.value").alias("indicator_name"),
        col("date").cast(IntegerType()).alias("year"),
        col("value").cast(DoubleType()).alias("value"),
        col("load_timestamp")
    )
    .withColumn("is_reported", when(col("value").isNotNull(), True).otherwise(False))
    .dropDuplicates(["country_code", "indicator_code", "year"])
)

silver_df.printSchema()
print(f"Silver row count: {silver_df.count()}")

root
 |-- country_code: string (nullable = true)
 |-- country_name: string (nullable = true)
 |-- indicator_code: string (nullable = true)
 |-- indicator_name: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- value: double (nullable = true)
 |-- load_timestamp: timestamp (nullable = false)
 |-- is_reported: boolean (nullable = false)

Silver row count: 302409


In [0]:
silver_df.write.format("delta").mode("overwrite").saveAsTable("silver_indicator_values")